#ENCODE THE OUTCOME VARIABLE & JOIN/MERGE WITH DEPRIVATION IMD SCORES

In [ ]:
# encode the last outcome category into binary resolved/unresolved outcomes

In [ ]:
STEPS TO TAKE

1. Read in the cleaned dataframe - saved as parquet this time
2. create copy of df for multinomial modelling later
3. remove ambiguous categories that are hard to justify
4. check 

In [ ]:
#import libraries

import pandas as pd

In [ ]:
# read in the dataframe

cleaned_data = pd.read_parquet("data/cleaned_data.parquet")

### Notes for ref.
#### cleaned_df -the CSV
#### cleaned data - the parquet
#### model_df_multiclass - the original non collapsed outcome category df
#### model_df_binary - the df with only 2 binary options in outcome

In [ ]:

# Take a copy of the df before collapsing to keep all original outcome categories for multinomial model later 

model_df_multiclass = cleaned_data.copy()

In [ ]:
# Remove the ambigious categories  rows  — the rows with those four outcome categories are filtered out below

unresolved = [
    'Status update unavailable',
    'Awaiting court outcome',
    'Court result unavailable',
    'Under investigation'
]

model_df_binary = model_df[~model_df['last_outcome_category'].isin(unresolved)].copy()

# sanity check
model_df_binary.shape, model_df_binary['last_outcome_category'].isin(unresolved).sum()

The next step below creates a new column outcome_binary (1 = resolved, 0 = unresolved) while leaving last_outcome_category untouched as the original text 

In [ ]:
# encode a binary version of the outcome category - using model_df_binary and outcome_binary is the new category

# Final classification, per the project's outcome category grouping table:
#   Resolved (1)   -> formal sanction or discretionary disposal by police
#   Unresolved (0) -> no actionable outcome for the victim
#   Excluded       -> pending/missing data, not a true outcome (dropped before modelling)

resolved_outcomes = [
    'Offender given a caution',
    'Suspect charged as part of another case',
    'Offender given a drugs possession warning',
    'Local resolution',
    'Action to be taken by another organisation',
    'Further action is not in the public interest',
    'Further investigation is not in the public interest',
    'Formal action is not in the public interest',
]

unresolved_outcomes = [
    'Unable to prosecute suspect',
    'Investigation complete; no suspect identified',
]

excluded_outcomes = [
    'Status update unavailable',
    'Awaiting court outcome',
    'Court result unavailable',
    'Under investigation',
]

# Drop excluded/pending categories BEFORE encoding, so they don't silently
# fall into the unresolved (0) class
model_df_binary = model_df_binary[
    ~model_df_binary['last_outcome_category'].isin(excluded_outcomes)
].copy()

# Sanity check: every remaining row should be classifiable as resolved or unresolved
unclassified = ~model_df_binary['last_outcome_category'].isin(resolved_outcomes + unresolved_outcomes)
if unclassified.any():
    print("Warning — unclassified categories found, review before proceeding:")
    print(model_df_binary.loc[unclassified, 'last_outcome_category'].value_counts())

model_df_binary['outcome_binary'] = model_df_binary['last_outcome_category'].apply(
    lambda x: 1 if x in resolved_outcomes else 0
)

# sanity check
print(model_df_binary['outcome_binary'].value_counts())
print(model_df_binary['outcome_binary'].value_counts(normalize=True).round(4) * 100)

In [ ]:
# sanity check 2

model_df_binary.head()

In [ ]:
#confirm the encoding logic actually mapped correctly. This should show category by category, which text label got mapped to 0 vs 1 

model_df_binary.groupby('last_outcome_category')['outcome_binary'].first()

In [ ]:
# MERGE/Join data to LSOA

In [ ]:
# merge or join the files on lsoa code


#merged = crime.merge(imd, left_on='LSOA code', right_on='LSOA code (2021)', how='left')